# PPO Training Smoke Test
> This notebook currently uses `MockDetector` only for integration testing. Final experimental results must use a real deepfake detector provided through the detector adapter interface.

## 1. Imports

In [ ]:
from pathlib import Path
import csv
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from PIL import Image

from src.ppo.environment import DeepfakeAttackEnv
from src.ppo.mock_detector import MockDetector
from src.ppo.train import create_ppo_model, load_ppo, run_episode, save_ppo, train_ppo

## 2. Create the mock-backed environment

In [ ]:
height, width = 96, 128
x = np.linspace(48, 240, width, dtype=np.uint8)
gradient = np.tile(x, (height, 1))
image = Image.fromarray(np.stack([gradient] * 3, axis=2)).convert('RGB')
env = DeepfakeAttackEnv(image, MockDetector(), max_steps=5, seed=42)

## 3. Create PPO

In [ ]:
model = create_ppo_model(env, seed=42)

## 4. Train for 1,000 requested timesteps

In [ ]:
model = train_ppo(model, total_timesteps=1000)

## 5. Save the checkpoint

In [ ]:
checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'ppo_mock_v1'
save_ppo(model, checkpoint_path)
print(checkpoint_path.with_suffix('.zip'))

## 6. Reload the checkpoint

In [ ]:
loaded_model = load_ppo(checkpoint_path, env)
print('Checkpoint reloaded.')

## 7. Run and save one deterministic episode

In [ ]:
history = run_episode(loaded_model, env, deterministic=True)
columns = ['step', 'action_name', 'before_confidence', 'after_confidence', 'reward', 'success']
for row in history:
    print({column: row[column] for column in columns})

results_path = PROJECT_ROOT / 'results' / 'ppo_mock_episode.csv'
results_path.parent.mkdir(parents=True, exist_ok=True)
with results_path.open('w', newline='') as output_file:
    writer = csv.DictWriter(output_file, fieldnames=history[0].keys())
    writer.writeheader()
    writer.writerows(history)
print(results_path)